# Word embeddings and Summary Model

**Information**


*** 
**Background information**



***
**Coding sources**

* 


***
**Aim of the code template**

Provide an example for a model call, which produces deterministic results.

## Get API key(s)

In [1]:
import os
import sys

# Assuming 'src' is one level down (in the current directory or a subdirectory)
path_to_src = os.path.join('src')  # Moves one level down to 'src' folder

# Add the path to sys.path
sys.path.append(path_to_src)

# Now you can import your API_key module
import API_key as key

In [2]:
embedding_model = 'Qwen/Qwen3-Embedding-0.6B'

# load local embedding model

In [3]:
# initalize embedding model:
from sentence_transformers import SentenceTransformer
import torch

if embedding_model == 'Qwen/Qwen3-Embedding-0.6B':
    if torch.cuda.is_available():
        model = SentenceTransformer(embedding_model, trust_remote_code=True).cuda()
    else:
        model = SentenceTransformer(
            embedding_model,
            trust_remote_code=True,
            config_kwargs={"use_memory_efficient_attention": False, "unpad_inputs": False}
        )
elif embedding_model == 'dunzhang/stella_en_400M_v5':
    model = SentenceTransformer(
    'dunzhang/stella_en_400M_v5',
    trust_remote_code=True,
    config_kwargs={"use_memory_efficient_attention": False, "unpad_inputs": False}
)
else:
    raise ValueError(f"Unsupported embedding model: {embedding_model}")

/home/fenn/Documents/env_python/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


nice printing:

In [4]:
from rich import print

# Example: extract word embeddings

## Feature Extraction with `sentence_transformers`

The following begins by extracting features (embeddings) from the text data---numerical representations of the meaning of text---using the `sentence_transformers` package.

The code makes use of the `dunzhang/stella_en_400M_v5` model, which is a larger embedding model, to extract features from the sentences. The model will encode the sentences into 1024-dimensional vector representations. The cell will then print the features as a pandas dataframe for easy viewing. See model page: https://huggingface.co/Marqo/dunzhang-stella_en_400M_v5

In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# Define sentences
sentences = [
    "I feel great this morning",
    "I am feeling very good today",
    "Ich fühle mich heute sehr gut",
    "I am feeling terrible",
]

# Extract features
features = model.encode(sentences)

# Print the features as a pandas dataframe
pd.DataFrame(features, index=sentences)

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
I feel great this morning,-0.036147,0.011035,-0.006297,0.013397,0.044439,-0.003271,-0.008532,0.035651,-0.064924,0.071769,...,-0.029760,-0.000770,0.004337,0.007651,0.004444,-0.025157,-0.005548,-0.002070,0.001456,-0.014095
I am feeling very good today,-0.025371,0.000649,-0.006473,0.002155,0.046450,-0.024363,-0.010254,0.023655,-0.042414,0.074530,...,-0.050013,0.007345,0.013080,0.017070,0.021635,-0.010721,-0.002800,-0.007240,0.009513,-0.008311
Ich fühle mich heute sehr gut,-0.016749,-0.005145,-0.003332,-0.006768,0.024511,-0.025672,-0.006922,0.031797,-0.034459,0.084131,...,-0.041056,-0.014588,-0.006112,0.014183,0.052204,-0.014012,-0.002646,-0.004214,0.016159,-0.017599
I am feeling terrible,0.001885,0.029234,-0.011215,0.017682,0.046844,-0.010958,-0.049366,-0.041810,-0.028118,0.050442,...,-0.054489,0.007802,-0.018711,0.016927,0.018709,-0.006085,0.011654,-0.037236,0.032078,0.020472


In [6]:
features.shape

(4, 1024)

In [7]:
similarities = model.similarity(features, features)
print(similarities)

tensor([[1.0000, 0.9002, 0.8105, 0.6852],
        [0.9002, 1.0000, 0.8876, 0.7259],
        [0.8105, 0.8876, 1.0000, 0.6676],
        [0.6852, 0.7259, 0.6676, 1.0000]])

# Load and prepare data


load data:

In [8]:
import pandas as pd

os.getcwd()

# Define the file paths relative to the current working directory
file_path_data = os.path.join('output', 'loop.xlsx')
file_path_definitions = os.path.join('data', 'laypersonDefinition_ethicTheories.xlsx')

# Load the Excel file into a pandas DataFrame
try:
    df = pd.read_excel(file_path_data)
    df_definitions = pd.read_excel(file_path_definitions)
    print("Files loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{file_path_data}' was not found.")
    print(f"Error: Or the file '{file_path_definitions}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Files loaded successfully.

In [9]:
df.shape

(644, 9)

In [10]:
df['ID'].value_counts()

ID
115    7
1      7
2      7
3      7
99     7
      ..
8      7
9      7
11     7
12     7
13     7
Name: count, Length: 92, dtype: int64

In [59]:
df['item'].value_counts()

item
hedonism01          40
deontology08        38
virtue09            35
deontology02        33
deontology01        33
contractualist03    33
deontology07        32
deontology04        32
contractualist06    32
hedonism07          32
hedonism08          32
hedonism03          30
deontology03        30
contractualist01    30
deontology05        29
deontology09        29
utilitarian01       28
virtue01            25
contractualist02    24
hedonism02          24
contractualist04    23
Name: count, dtype: int64

In [11]:
df.head()

,ID,item,questionItem,value,category_probe,category_probe_again,questionComprehension,comp_probe,comp_probe_again
0,1,hedonism01,is personally unsatisfactory...is personally s...,4,I selected 4 on this scale because I am torn b...,NaN,When in your opinion is the described technolo...,"In my opinion, the described technology is per...",NaN
1,1,hedonism03,requires me to make sacrifices in order to use...,7,I selected 7 for this scale because if it is i...,NaN,When in your opinion requires the described te...,"In my opinion, the described technology does n...",NaN
2,1,contractualist02,is unfair...is fair,6,I selected 6 because it has the potential to b...,NaN,When in your opinion is the described technolo...,In my opinion I described the technology as fa...,NaN
3,1,deontology02,harms the autonomy of users...promotes the aut...,4,I selected 4 in this scale because I believe i...,NaN,What does <b>autonomy of users</b> mean for yo...,Autonomy of users in a technological context m...,NaN
4,1,deontology03,obliges a certain immoral behavior...obliges a...,6,I chose 6 on this scale because I can agree it...,NaN,What do you consider to be a <b>moral behavior...,Moral behavior in a technological context woul...,NaN


In [12]:
df_definitions.head()

,ethicTheory,definition
0,Deontology,In the following we want to ask you to evaluat...
1,Utilitarianism,In the following we want to ask you to evaluat...
2,Hedonism,In the following we want to ask you to evaluat...
3,Virtue ethics,In the following we want to ask you to evaluat...
4,Contractualism,In the following we want to ask you to evaluat...


# Summarize answers

## category selection probe


*Set up prompting:*

In [ ]:
# Prompt for a bio-inspired technologies researcher
system_content = '''
You are a cognitive science expert specializing in cognitive interviews, with a focus on analyzing how respondents structure their arguments 
during web probing surveys related to technology ethics assessments. Your expertise lies in understanding how individuals align their ethical 
evaluations and concerns with specific technologies, using a variety of moral frameworks, including utilitarianism, virtue ethics, contractualism, 
deontology, and hedonism.

You are proficient in applying cognitive probing methods to uncover the reasoning behind participants' responses in ethical dilemmas surrounding 
emerging technologies. Your primary task is to systematically structure and categorize participants' responses to category selection probes. 
In doing so, you will ensure that their reasoning is clearly mapped to the appropriate ethical theory, and the underlying arguments are carefully 
analyzed. 

A **category selection probe** is a tool used to prompt participants to explain why they selected a particular rating, encouraging them 
to articulate the ethical principles or concerns that guided their decision. This helps identify their reasoning processes.

Throughout the process, maintain a neutral stance and employ a theory-driven approach, adhering to best practices in empirical ethics and cognitive 
interview methodology. This structured analysis will help uncover how participants conceptualize and justify their ethical evaluations of new technologies.
'''

user_content = '''
Please analyze the responses to a category selection probe regarding ethical evaluations of the technology "Stratospheric Aerosol Injection." 
Participants were instructed as follows:

"Please indicate on the following scale how you would rate the ethical implications of the technology described in the scenario text. 
The technology Stratospheric Aerosol Injection..."
{item_question}

The participants provided responses, each consisting of a rating (from 1 to 7) and a free-text explanation. The responses are as follows:
{merged_responses_str}

Your task is to categorize the responses into thematic groups, considering their given ratings, based on ethical theories. 
Ensure that each response is clearly linked to the following ethical theory: {theory}.

### Instructions:
1. **Categorize Each Response**: 
   - Read each participant’s response carefully. 
   - Identify the key ethical concern raised in their explanation.
   - Link the response to the ethical theory: {theory}. 
   
2. **Thematic Grouping**:
   - Organize responses into themes that reflect the ethical concerns.
   - Group similar responses under the same theme.

3. **Provide Scientific Explanations**:
   - For each theme, provide a concise, scientifically grounded explanation of why it belongs to that ethical theory.
   - Explain the reasoning clearly and link it to the core tenets of the ethical theory of {theory}.

4. **Return Key Themes**:
   - Only return the **emergent key themes**. Do not provide unnecessary commentary or additional explanations.
   - Present the themes in **bullet points** for clarity.

5. **Neutrality**:
   - Maintain a neutral stance when categorizing. Do not inject personal interpretation or biases into the analysis.
   
Provide no further commentary.
'''

*call model:*

In [14]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=key.hugging_api_key,
)

In [54]:
sorted(df["item"].unique())

['contractualist01',
 'contractualist02',
 'contractualist03',
 'contractualist04',
 'contractualist06',
 'deontology01',
 'deontology02',
 'deontology03',
 'deontology04',
 'deontology05',
 'deontology07',
 'deontology08',
 'deontology09',
 'hedonism01',
 'hedonism02',
 'hedonism03',
 'hedonism07',
 'hedonism08',
 'utilitarian01',
 'virtue01',
 'virtue09']

In [15]:
ethical_theories = list({''.join([c for c in item if not c.isdigit()]) for item in df["item"].unique()})
ethical_theories

['deontology', 'utilitarian', 'hedonism', 'virtue', 'contractualist']

In [ ]:
array_results = []


for(i, theory) in enumerate(ethical_theories):
    print(f"{i+1}. {theory}")
    tmp_df = df[df['item'].str.contains(theory)]
    tmp_df = tmp_df.sort_values(by='item')

    ethical_theories_items = tmp_df['item'].unique()
    print(f"Number of items for {theory} ethics: {len(ethical_theories_items)}, items: {ethical_theories_items}")

    for(j, item) in enumerate(ethical_theories_items):
        print(f"  {j+1}. {item}")
        item_df = tmp_df[tmp_df['item'] == item]
        # print(f"    Number of responses: {len(item_df)}")

        item_question = item_df["questionItem"].unique()[0]
        item_question = f"{item_question.split('...')[0]} (1) <-> {item_question.split('...')[1]} (7)"

        # Overwrite 'category_probe' with 'category_probe_again' where notna
        item_df.loc[item_df['category_probe_again'].notna(), 'category_probe'] = item_df.loc[item_df['category_probe_again'].notna(), 'category_probe_again']

        # Merge 'value' and 'category_probe' into a new column 'merged_response'
        item_df['merged_response'] = item_df.apply(lambda row: f"value: {row['value']}, response: {row['category_probe']}", axis=1)


        merged_responses_str = "\n".join(item_df['merged_response'].tolist())
        # print(merged_responses_str)

        # Call the Large Language Model (LLM) with the formatted prompt:
        formatted_user_content = user_content.format(item_question=item_question, merged_responses_str=merged_responses_str, theory=theory)

        completion = client.chat.completions.create(
        model="meta-llama/Llama-3.3-70B-Instruct:together",
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": formatted_user_content}
        ],
        stream=False,
        max_tokens=5000,
        temperature=0
    )

        array_results.append({
            "theory": theory,
            "item": item,
            "item_question": item_question,
            "LLM": completion.choices[0].message.content
        })

In [ ]:
# ...existing code...
import pandas as pd
import os

# Convert array_results to DataFrame
df_results = pd.DataFrame(array_results)

# Ensure output directory exists
output_dir = os.path.join('output', 'LLM')
os.makedirs(output_dir, exist_ok=True)

# Save as CSV
csv_path = os.path.join(output_dir, 'categorySelectionProbe.csv')
df_results.to_csv(csv_path, index=False)

# Save as XLSX
xlsx_path = os.path.join(output_dir, 'categorySelectionProbe.xlsx')
df_results.to_excel(xlsx_path, index=False)

print(f"Results saved to:\n{csv_path}\n{xlsx_path}")

Results saved to:
outputs/LLM/categorySelectionProbe.csv
outputs/LLM/categorySelectionProbe.xlsx

## comprehension probe


*Set up prompting:*

In [ ]:
system_content = '''
You are a cognitive science expert specializing in cognitive interviews, with a focus on analyzing how respondents structure their arguments 
during web probing surveys related to technology ethics assessments. Your expertise lies in understanding how individuals align their ethical 
evaluations and concerns with specific technologies, using a variety of moral frameworks, including utilitarianism, virtue ethics, contractualism, 
deontology, and hedonism.

You are proficient in applying cognitive probing methods to uncover the reasoning behind participants' responses in ethical dilemmas surrounding 
emerging technologies. Your primary task is to systematically structure and categorize participants' responses to comprehension probes. 
In doing so, you will ensure that their reasoning is clearly mapped to the appropriate ethical theory, and the underlying arguments are carefully 
analyzed. 

A **comprehension probe** is a tool used to prompt participants to assess how they interpret key terms or concepts. This helps identify their reasoning processes.

Throughout the process, maintain a neutral stance and employ a theory-driven approach, adhering to best practices in empirical ethics and cognitive 
interview methodology. This structured analysis will help uncover how participants conceptualize and justify their ethical evaluations of new technologies.
'''

user_content = '''
Please analyze the responses to a comprehension probe regarding ethical evaluations of the technology "Stratospheric Aerosol Injection." 
Participants were instructed as follows:

{item_comprehension}
In doing so, refer to the question: The technology Stratospheric Aerosol Injection "{item_question}"

The participants provided responses, in form of a free-text explanation. The responses are as follows:
{merged_responses_str}

Your task is to categorize the responses into thematic groups based on ethical theories. 
Ensure that each response is clearly linked to the following ethical theory: {theory}.


### Instructions:
1. **Categorize Each Response**: 
   - Read each participant’s response carefully. 
   - Identify the key ethical concern raised in their explanation.
   - Link the response to the ethical theory: {theory}, and the comprehension probe: {item_comprehension}.
   
2. **Thematic Grouping**:
   - Organize responses into themes that reflect the ethical concerns.
   - Group similar responses under the same theme.

3. **Provide Scientific Explanations**:
   - For each theme, provide a concise, scientifically grounded explanation of why it belongs to that ethical theory.
   - Explain the reasoning clearly and link it to the core tenets of the ethical theory of {theory}, and the comprehension probe: {item_comprehension}.

4. **Return Key Themes**:
   - Only return the **emergent key themes**. Do not provide unnecessary commentary or additional explanations.
   - Present the themes in **bullet points** for clarity.

5. **Neutrality**:
   - Maintain a neutral stance when categorizing. Do not inject personal interpretation or biases into the analysis.
   
Provide no further commentary.
'''

*call model:*

In [ ]:
array_results = []


for(i, theory) in enumerate(ethical_theories):
    print(f"{i+1}. {theory}")
    tmp_df = df[df['item'].str.contains(theory)]
    tmp_df = tmp_df.sort_values(by='item')

    ethical_theories_items = tmp_df['item'].unique()
    print(f"Number of items for {theory} ethics: {len(ethical_theories_items)}, items: {ethical_theories_items}")

    for(j, item) in enumerate(ethical_theories_items):
        print(f"  {j+1}. {item}")
        item_df = tmp_df[tmp_df['item'] == item]
        # print(f"    Number of responses: {len(item_df)}")

        item_comprehension = item_df["questionComprehension"].unique()[0]

        item_question = item_df["questionItem"].unique()[0]
        item_question = f"{item_question.split('...')[0]} (1) <-> {item_question.split('...')[1]} (7)"

        # Overwrite 'comp_probe' with 'comp_probe_again' where notna
        item_df.loc[item_df['comp_probe_again'].notna(), 'comp_probe'] = item_df.loc[item_df['comp_probe_again'].notna(), 'comp_probe_again']

        # Merge 'value' and 'category_probe' into a new column 'merged_response'
        item_df['merged_response'] = item_df.apply(lambda row: f"response: {row['comp_probe']}", axis=1)


        merged_responses_str = "\n".join(item_df['merged_response'].tolist())
        # print(merged_responses_str)

        # Call the Large Language Model (LLM) with the formatted prompt:
        formatted_user_content = user_content.format(item_comprehension=item_comprehension, item_question=item_question, 
                                                     merged_responses_str=merged_responses_str, theory=theory)

        completion = client.chat.completions.create(
        model="meta-llama/Llama-3.3-70B-Instruct:together",
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": formatted_user_content}
        ],
        stream=False,
        max_tokens=5000,
        temperature=0
    )

        array_results.append({
            "theory": theory,
            "item": item,
            "item_question": item_question,
            "LLM": completion.choices[0].message.content
        })

In [40]:
merged_responses_str

"response: My idea of fairness is making sure everyone knows of the side effects and for everyone to vote on if it should be used, and majority vote.\nresponse: I guess having a choice is what I think of.\nresponse: Fair use of technology is a use of technology that balances requirements with costs.\nresponse: I believe fairness in this context means that we do not lose anything in the interaction. Therefore, it's a fair transaction between man and technology.\nresponse: I think fairness would be how people benefit from it (for better or for worse). If a technology is only provided to a select few who could benefit from it, I think that would be unfair. If it's something everyone could benefit from (like SAI), then I consider it to be fair. If it's something forced upon people and could harm them, I would consider that unfair. So, if hypothetically SAI were actually harmful and people had no say in whether they had to be exposed to it, I would say that is unfair.\nresponse: An idea of 

In [36]:
array_results

[{'theory': 'deontology',
  'item': 'deontology01',
  'item_question': 'does not imply a moral obligation to act in a certain way (1) <-> implies a moral obligation to act in a certain way (7)',
  'LLM': '* **Duty to Act**: Responses emphasizing a moral obligation to act in a certain way, such as "We are on earth to protect creation", "A moral obligation is a commitment to ethics and the overall wellbeing of all individuals", and "It is a moral obligation to correct the global ecological damage caused by human carbon dioxide emissions to prevent ecological collapse", reflect a deontological perspective, where moral obligations are based on duties and rules, regardless of consequences.\n* **Right vs. Wrong**: Responses focusing on distinguishing right from wrong, such as "Doing what you feel is right, that is what i think this means", "I consider a moral obligation would be for humans to act in a way in which they know right from wrong and choose the right way", and "To do something tha

In [ ]:
# ...existing code...
import pandas as pd
import os

# Convert array_results to DataFrame
df_results = pd.DataFrame(array_results)

# Ensure output directory exists
output_dir = os.path.join('output', 'LLM')
os.makedirs(output_dir, exist_ok=True)

# Save as CSV
csv_path = os.path.join(output_dir, 'comprehensionProbe.csv')
df_results.to_csv(csv_path, index=False)

# Save as XLSX
xlsx_path = os.path.join(output_dir, 'comprehensionProbe.xlsx')
df_results.to_excel(xlsx_path, index=False)

print(f"Results saved to:\n{csv_path}\n{xlsx_path}")

Results saved to:
outputs/LLM/comprehensionProbe.csv
outputs/LLM/comprehensionProbe.xlsx

### check for problems in understanding

*Set up prompting:*

In [46]:
system_content = '''
You are a cognitive science expert specializing in cognitive interviews, with a focus on analyzing how respondents interpret, understand, 
and clarify key concepts in ethical evaluations during web probing surveys. Your expertise lies in assessing how laypersons align their ethical 
evaluations with specific technologies, using moral frameworks such as utilitarianism, virtue ethics, contractualism, deontology, and hedonism.

Your primary task is to analyze participants' responses to comprehension probes. These probes are designed to assess how clearly participants 
understand key terms, concepts, and ethical implications within the context of new technologies. You will identify whether respondents clearly 
grasp the intended meaning of the items, how they interpret the ethical implications, and whether their responses align with the key ethical theories.

A **comprehension probe** is a tool used to prompt participants to explain how they interpret specific terms or concepts in the context of ethical assessments.
It helps identify their reasoning processes, ensuring that responses are grounded in the correct ethical frameworks.

Throughout this process, you must maintain a neutral, theory-driven approach, adhering to best practices in cognitive interviews and empirical ethics.
Your analysis will help determine how well participants understand and process the ethical implications of emerging technologies.
'''



user_content = '''
Please analyze the responses to a comprehension probe regarding ethical evaluations of the technology "Stratospheric Aerosol Injection." 
Participants were instructed as follows:

{item_comprehension}
In doing so, refer to the question: "The technology Stratospheric Aerosol Injection: {item_question}"

The participants provided responses in the form of a free-text explanation. The responses are as follows:
{merged_responses_str}

Your task is to evaluate **how well each participant understands** the key concepts and terms in the comprehension probe, focusing on **interpretability**, **clarity**, and **relevance**.

### Instructions:
1. **Assess Understanding**:
   - Review each participant’s response to determine how clearly they understand the key terms and concepts in the comprehension probe.
   - Evaluate if the participant has interpreted the ethical implications correctly and aligned their response with the intended meaning of the item.
   - Identify any **misunderstandings**, **ambiguities**, or **clarity issues** in their responses.
   - Ensure that the response aligns with the core concepts being assessed in the comprehension probe: {item_comprehension}.

2. **Neutrality**:
   - Maintain a neutral stance while assessing the responses. Do not inject personal interpretation or biases into the analysis.
   - Focus strictly on the participant's understanding of the terms and concepts.

Provide no further commentary.
'''


*call model:*

In [58]:
# array_results = []

for(i, theory) in enumerate(ethical_theories):
    print(f"{i+1}. {theory}")
    tmp_df = df[df['item'].str.contains(theory)]
    tmp_df = tmp_df.sort_values(by='item')

    ethical_theories_items = tmp_df['item'].unique()
    print(f"Number of items for {theory} ethics: {len(ethical_theories_items)}, items: {ethical_theories_items}")

    if theory != 'contractualist':
        print("Skipping...")
        continue

    for(j, item) in enumerate(ethical_theories_items):
        print(f"  {j+1}. {item}")
        item_df = tmp_df[tmp_df['item'] == item]
        # print(f"    Number of responses: {len(item_df)}")

        item_comprehension = item_df["questionComprehension"].unique()[0]

        item_question = item_df["questionItem"].unique()[0]
        item_question = f"{item_question.split('...')[0]} (1) <-> {item_question.split('...')[1]} (7)"

        # Overwrite 'comp_probe' with 'comp_probe_again' where notna
        item_df.loc[item_df['comp_probe_again'].notna(), 'comp_probe'] = item_df.loc[item_df['comp_probe_again'].notna(), 'comp_probe_again']

        # Merge 'value' and 'category_probe' into a new column 'merged_response'
        item_df['merged_response'] = item_df.apply(lambda row: f"response: {row['comp_probe']}", axis=1)


        merged_responses_str = "\n".join(item_df['merged_response'].tolist())
        # print(merged_responses_str)

        # Call the Large Language Model (LLM) with the formatted prompt:
        formatted_user_content = user_content.format(item_comprehension=item_comprehension, item_question=item_question, 
                                                     merged_responses_str=merged_responses_str, theory=theory)

        completion = client.chat.completions.create(
        model="meta-llama/Llama-3.3-70B-Instruct:together",
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": formatted_user_content}
        ],
        stream=False,
        max_tokens=5000,
        temperature=0
    )

        array_results.append({
            "theory": theory,
            "item": item,
            "item_question": item_question,
            "LLM": completion.choices[0].message.content
        })

1. deontology

Number of items for deontology ethics: 8, items: ['deontology01' 'deontology02' 'deontology03' 'deontology04'
 'deontology05' 'deontology07' 'deontology08' 'deontology09']

Skipping...

2. utilitarian

Number of items for utilitarian ethics: 1, items: ['utilitarian01']

...

/tmp/ipykernel_485770/791891266.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  item_df['merged_response'] = item_df.apply(lambda row: f"response: {row['comp_probe']}", axis=1)


4. contractualist04

/tmp/ipykernel_485770/791891266.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  item_df['merged_response'] = item_df.apply(lambda row: f"response: {row['comp_probe']}", axis=1)


5. contractualist06

/tmp/ipykernel_485770/791891266.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  item_df['merged_response'] = item_df.apply(lambda row: f"response: {row['comp_probe']}", axis=1)


In [60]:
array_results

[{'theory': 'deontology',
  'item': 'deontology01',
  'item_question': 'does not imply a moral obligation to act in a certain way (1) <-> implies a moral obligation to act in a certain way (7)',
  'LLM': "### Assessment of Participant Understanding\n\n1. **Response: an obligation to someone's morals**\n   - **Understanding:** Limited\n   - **Interpretability:** The response is vague and does not clearly define what a moral obligation is.\n   - **Clarity:** Low\n   - **Relevance:** Partially relevant, as it mentions morals but does not fully capture the concept of moral obligation.\n   - **Misunderstandings/Ambiguities:** The response implies a moral obligation is about someone else's morals, rather than a personal or societal commitment to ethical behavior.\n\n2. **Response: We are on earth to protect creation**\n   - **Understanding:** Partial\n   - **Interpretability:** The response implies a sense of responsibility but does not directly define what a moral obligation is.\n   - **Cla

In [61]:
# ...existing code...
import pandas as pd
import os

# Convert array_results to DataFrame
df_results = pd.DataFrame(array_results)

# Ensure output directory exists
output_dir = os.path.join('output', 'LLM')
os.makedirs(output_dir, exist_ok=True)

# Save as CSV
csv_path = os.path.join(output_dir, 'comprehensionProbe_understanding.csv')
df_results.to_csv(csv_path, index=False)

# Save as XLSX
xlsx_path = os.path.join(output_dir, 'comprehensionProbe_understanding.xlsx')
df_results.to_excel(xlsx_path, index=False)

print(f"Results saved to:\n{csv_path}\n{xlsx_path}")

Results saved to:
output/LLM/comprehensionProbe_understanding.csv
output/LLM/comprehensionProbe_understanding.xlsx